# Energy Price Forecasting and Drift Monitoring

This notebook walks through hourly electricity price forecasting using leakage-safe feature engineering, temporal validation, model comparison, and distribution-shift monitoring.

**Note:** The default dataset is synthetic. Results demonstrate methodology and do not represent real market performance.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from energy_forecasting.config import DEFAULT_RAW_DATA
from energy_forecasting.data import load_energy_data, temporal_split
from energy_forecasting.evaluate import compute_metrics
from energy_forecasting.features import build_features, get_feature_columns, prepare_xy
from energy_forecasting.pipeline import ensure_data
from energy_forecasting.train import train_all_models, cross_validate_model
from energy_forecasting.drift import run_drift_analysis

%matplotlib inline

## Objective

Forecast the next-hour electricity price from historical prices and optional exogenous variables (load, temperature). The workflow emphasizes past-only features, chronological splits, baseline comparison, and drift monitoring.

In [ ]:
raw_path = ensure_data(DEFAULT_RAW_DATA)
df, report = load_energy_data(raw_path)
print(f"Rows: {report.n_rows:,}")
print(f"Range: {report.start_timestamp} to {report.end_timestamp}")
print(f"Missing intervals: {report.missing_intervals}")
df.head()

## Exploratory analysis

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
sample = df.iloc[:168 * 4]
axes[0].plot(sample["timestamp"], sample["price"])
axes[0].set_title("Price — four-week sample")
axes[0].set_ylabel("Price")

if "load" in df.columns:
    axes[1].plot(sample["timestamp"], sample["load"], color="green")
    axes[1].set_ylabel("Load")
axes[1].set_xlabel("Timestamp")
plt.tight_layout()
plt.show()

df["price"].describe()

## Temporal split and leakage risks

Data is split chronologically: 70% train, 15% validation, 15% test.

Lag features use `.shift()`. Rolling statistics use `price.shift(1)`. Preprocessing is fit only on training data.

In [ ]:
featured = build_features(df)
train_df, val_df, test_df = temporal_split(featured)
feature_cols = get_feature_columns(featured)

x_train, y_train = prepare_xy(train_df, feature_cols)
x_val, y_val = prepare_xy(val_df, feature_cols)
x_test, y_test = prepare_xy(test_df, feature_cols)

print(f"Train: {len(x_train):,}  Validation: {len(x_val):,}  Test: {len(x_test):,}")

## Model comparison (validation)

In [ ]:
fitted, val_metrics, best_name = train_all_models(x_train, y_train, x_val, y_val)
pd.DataFrame({m: v for m, v in val_metrics.items()}).T.sort_values("mae")

In [ ]:
pd.DataFrame(cross_validate_model(fitted[best_name], x_train, y_train))

## Test-set evaluation

The best model is chosen using validation MAE only. Test metrics are computed once below.

In [ ]:
best_model = fitted[best_name]
test_preds = best_model.predict(x_test)
test_metrics = compute_metrics(y_test.values, test_preds)
print(f"Selected model: {best_name}")
print(f"Test MAE:   {test_metrics['mae']:.4f}")
print(f"Test RMSE:  {test_metrics['rmse']:.4f}")
print(f"Test MAPE:  {test_metrics['mape']:.4f}%")
print(f"Test sMAPE: {test_metrics['smape']:.4f}%")

In [ ]:
test_timestamps = test_df.loc[x_test.index, "timestamp"]
fig, axes = plt.subplots(2, 1, figsize=(14, 8))
axes[0].plot(test_timestamps, y_test.values, label="Actual", alpha=0.8)
axes[0].plot(test_timestamps, test_preds, label="Predicted", alpha=0.8)
axes[0].set_title("Test period: actual vs predicted")
axes[0].legend()

residuals = y_test.values - test_preds
axes[1].hist(residuals, bins=40, edgecolor="black", alpha=0.7)
axes[1].set_title("Test residual distribution")
axes[1].set_xlabel("Residual")
plt.tight_layout()
plt.show()

## Drift detection

Drift analysis compares reference and monitoring windows. Signals should trigger investigation, not automatic retraining.

In [ ]:
n = len(featured)
ref = featured.iloc[n - 1440 : n - 720]
mon = featured.iloc[n - 720 :]
x_ref, y_ref = prepare_xy(ref, feature_cols)
x_mon, y_mon = prepare_xy(mon, feature_cols)

drift_report = run_drift_analysis(
    ref.loc[x_ref.index], mon.loc[x_mon.index], feature_cols,
    reference_preds=best_model.predict(x_ref),
    monitoring_preds=best_model.predict(x_mon),
    reference_labels=y_ref.values,
    monitoring_labels=y_mon.values,
)
print(drift_report["summary"])

## Limitations and next steps

- Synthetic data is used by default.
- Real markets depend on rules, generation mix, congestion, outages, and fuel prices.
- Strong synthetic performance does not guarantee real-market performance.
- A full artifact export (figures, JSON metrics, saved model) is produced by `python scripts/run_training.py`.